# osu!mania Difficulty Model - Colab Training

Use this when local training is too slow. Set `REPO_URL`, enable a GPU runtime, then run top to bottom.


In [ ]:
# Setup
REPO_URL = "https://github.com/fatelvx/ODP-model.git"
BRANCH = ""  # Optional. Leave empty to use the repo default branch.

import os
import subprocess
from pathlib import Path

if not REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repo URL first.")

!rm -rf /content/mania-difficulty-model
clone_cmd = ["git", "clone", "--depth", "1"]
if BRANCH:
    clone_cmd += ["--branch", BRANCH]
clone_cmd += [REPO_URL, "/content/mania-difficulty-model"]
subprocess.run(clone_cmd, check=True)
%cd /content/mania-difficulty-model
!pip install -e .


In [ ]:
# Check GPU
import torch
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
!nvidia-smi


## Smoke Test

Run this first. It proves the training/report pipeline works before spending API calls or GPU time on real data.


In [ ]:
!python -m mania_difficulty.tools.make_synthetic_dataset --maps 128 --out data/processed/synthetic_colab
!python -m mania_difficulty.train \
  --labels data/processed/synthetic_colab/labels.csv \
  --sequences data/processed/synthetic_colab/sequences \
  --epochs 12 \
  --batch-size 32 \
  --run-name colab_synthetic \
  --model lstm \
  --device cuda


In [ ]:
from IPython.display import Image, display, HTML

run_dir = Path("outputs/runs/colab_synthetic")
display(Image(str(run_dir / "learning_curve.png")))
display(Image(str(run_dir / "prediction_scatter.png")))
display(HTML((run_dir / "run_report.html").read_text(encoding="utf-8")))


## Real osu! Data

Use this after the smoke test. The credentials live only in this Colab session.


In [ ]:
import getpass
os.environ["OSU_CLIENT_ID"] = getpass.getpass("OSU_CLIENT_ID: ")
os.environ["OSU_CLIENT_SECRET"] = getpass.getpass("OSU_CLIENT_SECRET: ")


In [ ]:
# Start smaller than 2000 while debugging. Raise --target after the pipeline is stable.
!python -m mania_difficulty.data.fetch_maps --target 300 --out data/raw/beatmaps.csv
!python -m mania_difficulty.data.fetch_osu_files --maps data/raw/beatmaps.csv --out-dir data/raw/osu
!python -m mania_difficulty.data.parse_notes --maps data/raw/beatmaps.csv --osu-dir data/raw/osu --out-dir data/processed/sequences
!python -m mania_difficulty.data.fetch_scores --maps data/raw/beatmaps.csv --out data/processed/labels.csv --min-scores 30


In [ ]:
!python -m mania_difficulty.train \
  --labels data/processed/labels.csv \
  --sequences data/processed/sequences \
  --epochs 50 \
  --batch-size 32 \
  --run-name colab_lstm_top100 \
  --model lstm \
  --device cuda


In [ ]:
from google.colab import files
!zip -r colab_lstm_top100_outputs.zip outputs/runs/colab_lstm_top100
files.download("colab_lstm_top100_outputs.zip")
